# Preliminary experiments: BO versus CMA-ES

Exploratory experiments 1–3B. These are historical pilots, not the authoritative final comparison. Run cells in order in a Colab environment; some cells are computationally expensive.

**Setup:** Google Colab with required Python packages and the project Google Drive folder. See the repository README for reproducibility instructions.


In [ ]:

# ============================================================
# BO vs CMA-ES: EXPERIMENT 1
# Four-function baseline and runtime pilot
# ============================================================

!pip -q install cma scikit-learn scipy pandas matplotlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import warnings
import cma

from scipy.optimize import minimize
from scipy.stats import qmc
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, ConstantKernel

warnings.filterwarnings("ignore", category=UserWarning)

# ---------------- CONFIGURATION ----------------

DIM = 5
BUDGET = 60
N_INITIAL = 10
SEEDS = [42, 123]
LOW, HIGH = -5.0, 5.0

# ---------------- OBJECTIVE FUNCTIONS ----------------

def sphere(x):
    return float(np.sum(x**2))

def ackley(x):
    d = len(x)
    return float(
        -20*np.exp(-0.2*np.sqrt(np.mean(x**2)))
        -np.exp(np.mean(np.cos(2*np.pi*x)))
        +20+np.e
    )

def rosenbrock(x):
    return float(np.sum(
        100*(x[1:]-x[:-1]**2)**2
        +(1-x[:-1])**2
    ))

def rastrigin(x):
    d = len(x)
    return float(
        10*d + np.sum(x**2-10*np.cos(2*np.pi*x))
    )

FUNCTIONS = {
    "Sphere": sphere,
    "Ackley": ackley,
    "Rosenbrock": rosenbrock,
    "Rastrigin": rastrigin
}

# Known global optimum values are zero.
# Rosenbrock optimum is at x = (1,...,1).
# Other three optima are at x = (0,...,0).

# ---------------- BAYESIAN OPTIMISATION ----------------

def run_bo(func, seed, initial_points):
    rng = np.random.default_rng(seed + 1000)

    X = [x.copy() for x in initial_points]
    y = [func(x) for x in X]

    history = list(np.minimum.accumulate(y))

    start = time.perf_counter()

    while len(y) < BUDGET:

        kernel = (
            ConstantKernel(1.0, constant_value_bounds="fixed")
            * Matern(
                length_scale=np.ones(DIM),
                length_scale_bounds="fixed",
                nu=2.5
            )
        )

        # Scale inputs to [0,1] and standardise outputs.
        X_scaled = (np.array(X)-LOW)/(HIGH-LOW)
        y_array = np.array(y)

        gp = GaussianProcessRegressor(
            kernel=kernel,
            alpha=1e-6,
            normalize_y=True,
            optimizer=None
        )

        gp.fit(X_scaled, y_array)

        best_y = np.min(y_array)

        # Lower confidence bound acquisition:
        # balances predicted value and uncertainty.
        def acquisition(z):
            z = np.asarray(z).reshape(1,-1)
            mean, std = gp.predict(z, return_std=True)
            return float(mean[0] - 2.0*std[0])

        # Multistart acquisition optimisation
        candidates = rng.uniform(
            0, 1, size=(8, DIM)
        )

        results = [
            minimize(
                acquisition,
                x0,
                method="L-BFGS-B",
                bounds=[(0,1)]*DIM,
                options={"maxiter":35}
            )
            for x0 in candidates
        ]

        z_next = min(results, key=lambda r:r.fun).x

        x_next = LOW + z_next*(HIGH-LOW)

        # Avoid duplicate points
        if np.min(np.linalg.norm(
            np.array(X)-x_next, axis=1
        )) < 1e-6:
            x_next = rng.uniform(LOW,HIGH,DIM)

        value = func(x_next)

        X.append(x_next)
        y.append(value)
        history.append(min(history[-1],value))

    elapsed = time.perf_counter()-start

    return {
        "best": min(y),
        "history": history,
        "time": elapsed,
        "best_x": np.array(X)[np.argmin(y)]
    }

# ---------------- CMA-ES ----------------

def run_cma(func, seed, initial_points):

    # Same starting region as BO, but CMA-ES does not
    # consume the initial design values. This is a
    # known pilot limitation to investigate later.
    x0 = np.mean(initial_points,axis=0)

    opts = {
        "seed":seed,
        "bounds":[LOW,HIGH],
        "verbose":-9,
        "verb_disp":0,
        "maxfevals":BUDGET,
        "popsize":8
    }

    es = cma.CMAEvolutionStrategy(
        x0, 2.0, opts
    )

    history = []
    best = np.inf
    best_x = None
    evaluations = 0

    start = time.perf_counter()

    while evaluations < BUDGET:
        candidates = es.ask()
        remaining = BUDGET-evaluations

        # Evaluate full generations when possible.
        # A final partial generation is evaluated
        # but not passed to tell().
        batch = candidates[:remaining]
        values = []

        for x in batch:
            val = func(np.asarray(x))
            values.append(val)

            if val < best:
                best = val
                best_x = np.asarray(x).copy()

            history.append(best)
            evaluations += 1

        if len(batch)==len(candidates):
            es.tell(candidates,values)
        else:
            break

        if es.stop():
            break

    elapsed = time.perf_counter()-start

    return {
        "best":best,
        "history":history,
        "time":elapsed,
        "best_x":best_x
    }

# ---------------- RUN EXPERIMENT ----------------

records = []
histories = {}

print("STARTING PILOT EXPERIMENT")
print("="*65)

for fname, func in FUNCTIONS.items():

    for seed in SEEDS:

        sampler = qmc.LatinHypercube(
            d=DIM,seed=seed
        )
        initial_points = qmc.scale(
            sampler.random(N_INITIAL),
            LOW,HIGH
        )

        for method,runner in [
            ("BO",run_bo),
            ("CMA-ES",run_cma)
        ]:

            try:
                result = runner(
                    func,seed,initial_points
                )

                records.append({
                    "function":fname,
                    "method":method,
                    "seed":seed,
                    "best_regret":result["best"],
                    "time_seconds":result["time"],
                    "evaluations":len(result["history"])
                })

                histories[(fname,method,seed)] = (
                    result["history"]
                )

                print(
                    f"{fname:12s} | "
                    f"{method:6s} | "
                    f"seed={seed} | "
                    f"regret={result['best']:.5f} | "
                    f"time={result['time']:.2f}s | "
                    f"evals={len(result['history'])}"
                )

            except Exception as e:
                print(
                    f"ERROR: {fname}, {method}, "
                    f"seed={seed}: {repr(e)}"
                )

df = pd.DataFrame(records)

print("\n" + "="*65)
print("SUMMARY: MEAN ACROSS SEEDS")
print("="*65)

if not df.empty:
    summary = df.groupby(
        ["function","method"]
    ).agg(
        mean_regret=("best_regret","mean"),
        mean_time=("time_seconds","mean"),
        mean_evals=("evaluations","mean")
    ).round(5)

    print(summary.to_string())

    df.to_csv(
        "experiment_1_pilot.csv",index=False
    )

    print("\nSaved: experiment_1_pilot.csv")

# ---------------- CONVERGENCE PLOTS ----------------

fig,axes = plt.subplots(
    2,2,figsize=(12,9)
)

for ax,(fname,func) in zip(
    axes.flat,FUNCTIONS.items()
):

    for method in ["BO","CMA-ES"]:

        curves = [
            histories[(fname,method,seed)]
            for seed in SEEDS
            if (fname,method,seed) in histories
        ]

        if not curves:
            continue

        for curve in curves:
            ax.plot(
                np.arange(1,len(curve)+1),
                np.maximum(curve,1e-10),
                alpha=0.65,
                label=method if curve is curves[0] else None
            )

    ax.set_title(fname)
    ax.set_xlabel("Objective evaluations")
    ax.set_ylabel("Best true regret")
    ax.set_yscale("log")
    ax.grid(alpha=0.25)
    ax.legend()

plt.suptitle(
    "Experiment 1: BO vs CMA-ES Baseline Pilot"
)
plt.tight_layout()
plt.show()

print("\nEXPERIMENT COMPLETE")


In [ ]:

# ============================================================
# EXPERIMENT 2A
# High-budget CMA-ES + GP computational scaling
# Requires Experiment 1 definitions
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import cma
import sklearn
import scipy

from scipy.stats import qmc
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, ConstantKernel

print("EXPERIMENT 2A: HIGH-BUDGET DIAGNOSTIC")
print("=" * 65)

MAX_BUDGET = 1000
CHECKPOINTS = [60, 150, 250, 400, 600, 1000]
TEST_SEEDS = [42, 123, 2026]

print("Python libraries:")
print("scikit-learn:", sklearn.__version__)
print("scipy:", scipy.__version__)
print("cma:", cma.__version__)

# ============================================================
# PART A: CMA-ES CONVERGENCE
# ============================================================

def run_cma_extended(func, seed, max_budget=1000):

    sampler = qmc.LatinHypercube(
        d=DIM, seed=seed
    )

    initial_points = qmc.scale(
        sampler.random(N_INITIAL),
        LOW, HIGH
    )

    x0 = np.mean(initial_points, axis=0)

    es = cma.CMAEvolutionStrategy(
        x0,
        2.0,
        {
            "seed": seed,
            "bounds": [LOW, HIGH],
            "popsize": 8,
            "verbose": -9,
            "verb_disp": 0,
            "maxfevals": max_budget,
            "tolfun": 0,
            "tolx": 1e-14,
            "tolstagnation": 100000
        }
    )

    best = float("inf")
    history = []
    eval_count = 0

    start = time.perf_counter()

    while eval_count < max_budget:

        candidates = es.ask()

        remaining = max_budget - eval_count
        batch = candidates[:remaining]

        values = []

        for x in batch:
            val = func(np.asarray(x))
            values.append(val)

            best = min(best, val)
            history.append(best)
            eval_count += 1

        if len(batch) == len(candidates):
            es.tell(candidates, values)
        else:
            break

        if es.stop():
            break

    elapsed = time.perf_counter() - start

    return np.asarray(history), elapsed


records = []
histories = {}

print("\nPART A: CMA-ES TO 1000 EVALUATIONS")
print("=" * 65)

for fname, func in FUNCTIONS.items():

    for seed in TEST_SEEDS:

        history, elapsed = run_cma_extended(
            func, seed, MAX_BUDGET
        )

        histories[(fname, seed)] = history

        for budget in CHECKPOINTS:

            if budget <= len(history):

                records.append({
                    "function": fname,
                    "seed": seed,
                    "budget": budget,
                    "regret": history[budget - 1],
                    "total_runtime": elapsed
                })

        print(
            f"{fname:12s} | seed={seed} | "
            f"evals={len(history)} | "
            f"final_regret={history[-1]:.6f} | "
            f"time={elapsed:.3f}s",
            flush=True
        )

df_budget = pd.DataFrame(records)

df_budget.to_csv(
    "experiment_2a_cma_budget.csv",
    index=False
)

print("\nCMA-ES: MEAN REGRET BY BUDGET")
print("=" * 65)

summary = df_budget.groupby(
    ["function", "budget"]
)["regret"].agg(
    ["mean", "std", "count"]
)

print(summary.round(6).to_string())

# ============================================================
# PART B: GAUSSIAN PROCESS FITTING TIME
# ============================================================

print("\nPART B: GP COMPUTATIONAL SCALING")
print("=" * 65)

GP_SIZES = [100, 250, 500, 750, 1000]

gp_records = []

rng = np.random.default_rng(42)

for n in GP_SIZES:

    X = rng.uniform(0, 1, (n, DIM))
    y = np.sum((X - 0.5)**2, axis=1)

    kernel = (
        ConstantKernel(
            1.0,
            constant_value_bounds="fixed"
        )
        * Matern(
            length_scale=np.ones(DIM),
            length_scale_bounds="fixed",
            nu=2.5
        )
    )

    gp = GaussianProcessRegressor(
        kernel=kernel,
        alpha=1e-6,
        normalize_y=True,
        optimizer=None
    )

    times = []

    for repeat in range(3):

        start = time.perf_counter()
        gp.fit(X, y)
        elapsed = time.perf_counter() - start

        times.append(elapsed)

    median_time = float(np.median(times))

    gp_records.append({
        "observations": n,
        "median_fit_seconds": median_time
    })

    print(
        f"GP observations={n:4d} | "
        f"median fitting time={median_time:.4f}s",
        flush=True
    )

df_gp = pd.DataFrame(gp_records)

df_gp.to_csv(
    "experiment_2a_gp_scaling.csv",
    index=False
)

# ============================================================
# PART C: PLOTS
# ============================================================

fig, axes = plt.subplots(
    2, 2, figsize=(12, 9)
)

for ax, fname in zip(
    axes.flat, FUNCTIONS.keys()
):

    for seed in TEST_SEEDS:

        h = histories[(fname, seed)]

        ax.plot(
            np.arange(1, len(h) + 1),
            np.maximum(h, 1e-12),
            alpha=0.7,
            label=f"Seed {seed}"
        )

    for b in CHECKPOINTS:
        ax.axvline(
            b, linestyle=":",
            color="gray", alpha=0.2
        )

    ax.set_title(fname)
    ax.set_xlabel("Objective evaluations")
    ax.set_ylabel("Best true regret")
    ax.set_yscale("log")
    ax.grid(alpha=0.2)
    ax.legend()

plt.suptitle(
    "CMA-ES: Convergence up to 1000 Evaluations"
)
plt.tight_layout()
plt.savefig(
    "experiment_2a_cma_convergence.png",
    dpi=160
)
plt.show()

# Separate GP scaling plot

plt.figure(figsize=(8, 5))

plt.plot(
    df_gp["observations"],
    df_gp["median_fit_seconds"],
    marker="o"
)

plt.xlabel("Number of observations")
plt.ylabel("Median GP fitting time (seconds)")
plt.title("Gaussian Process Computational Scaling")
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(
    "experiment_2a_gp_scaling.png",
    dpi=160
)
plt.show()

print("\nEXPERIMENT 2A COMPLETE")
print("Saved:")
print("experiment_2a_cma_budget.csv")
print("experiment_2a_gp_scaling.csv")
print("experiment_2a_cma_convergence.png")
print("experiment_2a_gp_scaling.png")


In [ ]:

# ============================================================
# EXPERIMENT 3A: NOISY RECOMMENDATION PILOT
# BO vs CMA-ES | 4 functions | 5 dimensions
# ============================================================

!pip -q install cma scikit-learn scipy pandas matplotlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import cma
import sklearn
import scipy

from scipy.stats import qmc
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern

# ---------------- CONFIG ----------------

D = 5
BUDGET = 150
N_INIT = 10
SEEDS = [42, 123]
NOISE_LEVELS = [0.0, 0.10]
LOW, HIGH = -5.0, 5.0

# ---------------- FUNCTIONS ----------------

def sphere(x):
    return float(np.sum(x**2))

def ackley(x):
    return float(
        -20*np.exp(-0.2*np.sqrt(np.mean(x**2)))
        -np.exp(np.mean(np.cos(2*np.pi*x)))
        +20+np.e
    )

def rosenbrock(x):
    return float(np.sum(
        100*(x[1:]-x[:-1]**2)**2
        +(1-x[:-1])**2
    ))

def rastrigin(x):
    return float(
        10*len(x)+np.sum(
            x**2-10*np.cos(2*np.pi*x)
        )
    )

FUNCTIONS3 = {
    "Sphere": sphere,
    "Ackley": ackley,
    "Rosenbrock": rosenbrock,
    "Rastrigin": rastrigin
}

# Estimate a reference objective scale independently
# of the optimisation runs.
# This is a noise-calibration convention, not a
# published universal standard.

rng_scale = np.random.default_rng(999)
reference_X = rng_scale.uniform(
    LOW, HIGH, (2000, D)
)

SCALES = {
    name: float(np.std([
        func(x) for x in reference_X
    ]))
    for name, func in FUNCTIONS3.items()
}

print("Reference scales:")
for name, scale in SCALES.items():
    print(name, round(scale, 4))

# ---------------- NOISY EVALUATOR ----------------

class Evaluator:

    def __init__(self, func, sigma, seed):
        self.func = func
        self.sigma = sigma
        self.rng = np.random.default_rng(seed)
        self.X = []
        self.y = []
        self.true_y = []

    def evaluate(self, x):
        x = np.asarray(x, dtype=float).copy()

        if not np.all(
            (x >= LOW-1e-8) & (x <= HIGH+1e-8)
        ):
            raise ValueError("Point outside domain")

        true_value = self.func(x)
        noisy_value = (
            true_value +
            self.rng.normal(0, self.sigma)
        )

        self.X.append(x)
        self.y.append(noisy_value)
        self.true_y.append(true_value)

        return noisy_value

    @property
    def count(self):
        return len(self.y)

# ---------------- SHARED INITIAL DESIGN ----------------

def initial_design(seed):
    sampler = qmc.LatinHypercube(
        d=D, seed=seed
    )
    return qmc.scale(
        sampler.random(N_INIT),
        LOW, HIGH
    )

# ---------------- BO ----------------

def run_bo3(func, sigma, seed):

    ev = Evaluator(func, sigma, seed+5000)
    rng = np.random.default_rng(seed+1000)

    X0 = initial_design(seed)

    for x in X0:
        ev.evaluate(x)

    start = time.perf_counter()

    while ev.count < BUDGET:

        X = np.asarray(ev.X)
        y = np.asarray(ev.y)

        Xs = (X-LOW)/(HIGH-LOW)

        ym = float(np.mean(y))
        ys = max(float(np.std(y)), 1e-8)

        yn = (y-ym)/ys

        kernel = Matern(
            length_scale=0.3,
            length_scale_bounds="fixed",
            nu=2.5
        )

        gp = GaussianProcessRegressor(
            kernel=kernel,
            alpha=max((sigma/ys)**2, 1e-8),
            optimizer=None,
            normalize_y=False
        )

        gp.fit(Xs, yn)

        # Candidate-pool lower confidence bound
        candidates = rng.uniform(
            0, 1, (256, D)
        )

        mu, std = gp.predict(
            candidates, return_std=True
        )

        acquisition = mu - 2.0*std
        chosen = candidates[np.argmin(acquisition)]

        x_next = LOW + chosen*(HIGH-LOW)
        ev.evaluate(x_next)

    # Final posterior-mean recommendation
    X = np.asarray(ev.X)
    y = np.asarray(ev.y)
    Xs = (X-LOW)/(HIGH-LOW)

    ym = float(np.mean(y))
    ys = max(float(np.std(y)), 1e-8)

    gp = GaussianProcessRegressor(
        kernel=Matern(
            length_scale=0.3,
            length_scale_bounds="fixed",
            nu=2.5
        ),
        alpha=max((sigma/ys)**2, 1e-8),
        optimizer=None
    )

    gp.fit(Xs, (y-ym)/ys)
    posterior_means = gp.predict(Xs)

    model_idx = int(np.argmin(posterior_means))
    noisy_idx = int(np.argmin(y))

    elapsed = time.perf_counter()-start

    return {
        "method": "BO",
        "evals": ev.count,
        "best_noisy_true": func(X[noisy_idx]),
        "model_true": func(X[model_idx]),
        "observed_min": float(y[noisy_idx]),
        "runtime": elapsed,
        "history": np.minimum.accumulate(
            ev.true_y
        )
    }

# ---------------- CMA-ES ----------------

def run_cma3(func, sigma, seed):

    ev = Evaluator(func, sigma, seed+5000)
    X0 = initial_design(seed)

    # Shared initial points are evaluated and charged.
    # CMA-ES uses their centroid but not their values.
    # This information asymmetry is explicitly retained
    # as a pilot limitation.

    for x in X0:
        ev.evaluate(x)

    x0 = np.mean(X0, axis=0)

    es = cma.CMAEvolutionStrategy(
        x0, 2.0,
        {
            "seed": seed,
            "bounds": [LOW, HIGH],
            "popsize": 8,
            "verbose": -9,
            "verb_disp": 0,
            "maxfevals": BUDGET,
            "tolfun": 0
        }
    )

    start = time.perf_counter()

    while ev.count < BUDGET:

        candidates = es.ask()
        remaining = BUDGET-ev.count

        batch = candidates[:remaining]

        values = [
            ev.evaluate(x)
            for x in batch
        ]

        if len(batch)==len(candidates):
            es.tell(candidates, values)
        else:
            break

        if es.stop():
            break

    X = np.asarray(ev.X)
    y = np.asarray(ev.y)

    noisy_idx = int(np.argmin(y))

    # Final CMA-ES distribution mean
    model_x = np.clip(
        np.asarray(es.mean), LOW, HIGH
    )

    elapsed = time.perf_counter()-start

    return {
        "method": "CMA-ES",
        "evals": ev.count,
        "best_noisy_true": func(X[noisy_idx]),
        "model_true": func(model_x),
        "observed_min": float(y[noisy_idx]),
        "runtime": elapsed,
        "history": np.minimum.accumulate(
            ev.true_y
        )
    }

# ---------------- EXPERIMENT ----------------

results3 = []
histories3 = {}

print("\nSTARTING EXPERIMENT 3A")
print("="*65)

for fname, func in FUNCTIONS3.items():

    for noise_fraction in NOISE_LEVELS:

        sigma = (
            noise_fraction*SCALES[fname]
        )

        for seed in SEEDS:

            for method, runner in [
                ("BO", run_bo3),
                ("CMA-ES", run_cma3)
            ]:

                print(
                    f"Running {fname}, {method}, "
                    f"noise={noise_fraction}, "
                    f"seed={seed}",
                    flush=True
                )

                result = runner(
                    func, sigma, seed
                )

                results3.append({
                    "function": fname,
                    "method": method,
                    "noise_fraction": noise_fraction,
                    "sigma": sigma,
                    "seed": seed,
                    "evals": result["evals"],
                    "best_noisy_true":
                        result["best_noisy_true"],
                    "model_true":
                        result["model_true"],
                    "observed_min":
                        result["observed_min"],
                    "runtime":
                        result["runtime"]
                })

                histories3[
                    (fname, method,
                     noise_fraction, seed)
                ] = result["history"]

                print(
                    f"  noisy-choice regret="
                    f"{result['best_noisy_true']:.5f}"
                    f" | state-choice regret="
                    f"{result['model_true']:.5f}"
                    f" | time={result['runtime']:.2f}s",
                    flush=True
                )

df3 = pd.DataFrame(results3)

df3.to_csv(
    "experiment_3a_noisy_pilot.csv",
    index=False
)

print("\nSUMMARY")
print("="*65)

summary3 = df3.groupby(
    ["function", "method", "noise_fraction"]
)[[
    "best_noisy_true",
    "model_true",
    "runtime"
]].mean().round(5)

print(summary3.to_string())

print("\nEVALUATION ACCOUNTING")
print(df3.groupby("method")["evals"].agg(
    ["min", "max"]
))

print("\nEXPERIMENT 3A COMPLETE")
print("Saved experiment_3a_noisy_pilot.csv")


In [ ]:

# ==========================================================
# EXPERIMENT 3B: REPLICATION VS EXPLORATION
# Fixed-budget candidate-selection mechanism study
# ==========================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import qmc

D = 5
B = 400
SEEDS = range(20)
FRACTIONS = [0.0, 0.10, 0.25, 0.50]
NOISE_LEVELS = [0.0, 0.05, 0.20]
K = 5

def sphere(x):
    return float(np.sum(x**2))

def ackley(x):
    return float(
        -20*np.exp(-0.2*np.sqrt(np.mean(x**2)))
        -np.exp(np.mean(np.cos(2*np.pi*x)))
        +20+np.e
    )

def rosenbrock(x):
    return float(np.sum(
        100*(x[1:]-x[:-1]**2)**2
        +(1-x[:-1])**2
    ))

def rastrigin(x):
    return float(
        10*len(x)+np.sum(
            x**2-10*np.cos(2*np.pi*x)
        )
    )

FUNCS = {
    "Sphere": sphere,
    "Ackley": ackley,
    "Rosenbrock": rosenbrock,
    "Rastrigin": rastrigin
}

# Use reproducible reference scales.
# These are the same calibration values as 3A.
SCALES = {
    "Sphere": 16.4726,
    "Ackley": 1.4384,
    "Rosenbrock": 35929.2781,
    "Rastrigin": 22.6677
}

def candidate_sequence(func, seed, n=B):
    """
    Candidate-generation mechanism:
    100 global Latin-hypercube points followed by
    local Gaussian perturbations around the best
    true-valued point in that initial design.

    IMPORTANT: This is an oracle-assisted diagnostic,
    NOT an implementable noisy optimiser.
    The same candidate sequence is shared by policies.
    """
    rng = np.random.default_rng(seed)

    sampler = qmc.LatinHypercube(
        d=D, seed=seed
    )
    global_x = qmc.scale(
        sampler.random(100), -5, 5
    )

    best_idx = np.argmin(
        [func(x) for x in global_x]
    )
    center = global_x[best_idx]

    local_x = np.clip(
        center + rng.normal(
            0, 1.0, size=(n-100,D)
        ),
        -5, 5
    )

    return np.vstack([global_x, local_x])


records = []

for fname, func in FUNCS.items():

    for noise_level in NOISE_LEVELS:

        sigma = noise_level*SCALES[fname]

        for seed in SEEDS:

            X = candidate_sequence(func, seed)
            true_values = np.array([
                func(x) for x in X
            ])

            # Common random numbers for the initial
            # candidate observations.
            rng = np.random.default_rng(
                seed + 100000
            )
            initial_noise = rng.normal(
                0, sigma, size=B
            )

            for fraction in FRACTIONS:

                n_validate = int(round(B*fraction))
                n_search = B-n_validate

                # First-stage exploration
                observed = (
                    true_values[:n_search]
                    + initial_noise[:n_search]
                )

                # Top K candidates from noisy observations
                top = np.argsort(observed)[:K]

                # Accumulate repeated observations
                sums = observed[top].copy()
                counts = np.ones(len(top),dtype=int)

                # Independent validation noise.
                # Seed is shared across fractions so
                # replication draws are reproducible.
                vrng = np.random.default_rng(
                    seed + 200000
                )

                for j in range(n_validate):
                    slot = j % len(top)
                    idx = top[slot]

                    new_y = (
                        true_values[idx]
                        + vrng.normal(0,sigma)
                    )

                    sums[slot] += new_y
                    counts[slot] += 1

                means = sums/counts
                selected = top[np.argmin(means)]

                # Oracle diagnostic: best true candidate
                # available within the search stage.
                oracle = np.min(
                    true_values[:n_search]
                )

                records.append({
                    "function":fname,
                    "noise_level":noise_level,
                    "seed":seed,
                    "fraction":fraction,
                    "search_evals":n_search,
                    "validation_evals":n_validate,
                    "total_evals":n_search+n_validate,
                    "selected_regret":true_values[selected],
                    "oracle_regret":oracle,
                    "selection_gap":
                        true_values[selected]-oracle
                })

df = pd.DataFrame(records)

assert (df["total_evals"] == B).all()
assert (df["selection_gap"] >= -1e-8).all()

df.to_csv(
    "experiment_3b_replication.csv",
    index=False
)

summary = df.groupby(
    ["function","noise_level","fraction"]
).agg(
    mean_regret=("selected_regret","mean"),
    median_regret=("selected_regret","median"),
    mean_selection_gap=("selection_gap","mean")
).reset_index()

print("\nEXPERIMENT 3B SUMMARY")
print(summary.round(5).to_string(index=False))

# Plot mean selection gap vs validation allocation
fig, axes = plt.subplots(2,2,figsize=(12,9))

for ax, fname in zip(
    axes.flat,FUNCS.keys()
):
    subset = summary[
        summary["function"]==fname
    ]

    for noise in NOISE_LEVELS:
        s = subset[
            subset["noise_level"]==noise
        ]

        ax.plot(
            s["fraction"]*100,
            s["mean_selection_gap"],
            marker="o",
            label=f"Noise={noise}"
        )

    ax.set_title(fname)
    ax.set_xlabel("Budget spent validating (%)")
    ax.set_ylabel("Mean selection gap")
    ax.grid(alpha=0.3)
    ax.legend()

plt.suptitle(
    "Experiment 3B: Validation Allocation"
)
plt.tight_layout()
plt.savefig(
    "experiment_3b.png",
    dpi=160
)
plt.show()

print("\nTotal scenarios:",len(df))
print("All budgets verified:",B)
print("Saved experiment_3b_replication.csv")
